# Exposure time vs z-position — manual XY scans, 2026-07-13

All manual scans taken 2026-07-13 used a per-z calibrated exposure (Shuo's idea:
maximize dynamic range at every z instead of one global exposure). This notebook:

1. loads every `manual_scan-2026-07-13_*` run and graphs **z-position vs calibrated exposure time**,
2. computes per-frame statistics (saturation, total counts) to sanity-check the
   *counts ∝ exposure-time* linearity assumption that any normalization relies on,
3. summarizes candidate normalization approaches (see discussion at the bottom).

Camera: FLIR BFS-PGE-31S4M, Mono8, Gain 0 dB, gamma disabled — so pixel counts
should be linear in (irradiance × exposure time) below saturation.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

DATA_ROOT = Path("../data")
SCAN_GLOB = "manual_scan-2026-07-13_*"

# The stage/tape-measure z readings are good to about +/-5 mm.
Z_ERR_CM = 0.5

# TODO: confirm the actual laser wavelength (also used by the QDHT model).
WAVELENGTH_NM = 650

runs = []
for run_dir in sorted(DATA_ROOT.glob(SCAN_GLOB)):
    setup_path = run_dir / "sweep_setup.json"
    if not setup_path.exists():
        print(f"skipping {run_dir.name}: no sweep_setup.json")
        continue

    setup = json.loads(setup_path.read_text())
    camera_settings = json.loads((run_dir / "camera_settings.json").read_text())

    runs.append(
        {
            "run_dir": run_dir,
            "z_cm": setup["SensorZ_cm"],
            "z_reference": setup["SensorZReference"],
            "exposure_us": setup["CalibratedExposure_us"],
            "pixel_format": camera_settings["PixelFormat"],
            "gain_db": camera_settings["Gain"],
            # exclude stitched composite.npy — raw frames only
            "frame_paths": sorted(
                p for p in run_dir.glob("*.npy")
                if not p.name.startswith("composite")
            ),
        }
    )

runs.sort(key=lambda r: (r["z_cm"], r["run_dir"].name))

# Optic configuration (identical across today's runs) — goes in plot titles.
OPTIC = json.loads(
    (runs[0]["run_dir"] / "sweep_setup.json").read_text()
)["OpticConfiguration"]
OPTIC_TITLE = (
    f"axicon #1,2 alpha={OPTIC['Axicon1_deg']:g} deg, "
    f"L12 separation={OPTIC['L12_mm']:.1f} mm, "
    f"axicon #3 alpha={OPTIC['Axicon3_deg']:g} deg, "
    f"L23 separation={OPTIC['L23_mm']:.1f} mm\n"
    f"input Gaussian waist {OPTIC['GaussianBeamWaist_mm']:.1f} mm, "
    f"wavelength {WAVELENGTH_NM:g} nm"
)
QDHT_ATTRIBUTION = "Quasi-Discrete Hankel Transform model (Yu et al. 1998)"

print(f"{len(runs)} runs")
print(f"{'z (cm)':>8}  {'exposure (us)':>14}  {'frames':>6}  run")
for r in runs:
    print(
        f"{r['z_cm']:>8.1f}  {r['exposure_us']:>14.1f}  "
        f"{len(r['frame_paths']):>6d}  {r['run_dir'].name}"
    )

# Every figure is also written out as images/<notebook filename>/<description>.png.
NOTEBOOK_STEM = "exposure_vs_z_analysis_2026_07_13"
IMAGES_DIR = Path("images") / NOTEBOOK_STEM
IMAGES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(description: str) -> None:
    """Save the current figure as images/<notebook>/<description>.png."""
    path = IMAGES_DIR / f"{description}.png"
    plt.gcf().savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")


In [ ]:
z_cm = np.array([r["z_cm"] for r in runs])
exposure_us = np.array([r["exposure_us"] for r in runs])
n_frames = np.array([len(r["frame_paths"]) for r in runs])

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.errorbar(
    z_cm, exposure_us, xerr=Z_ERR_CM,
    fmt="none", ecolor="0.55", elinewidth=1.2, capsize=2, zorder=2,
)
scatter = ax.scatter(
    z_cm,
    exposure_us,
    c=n_frames,
    cmap="viridis",
    s=55,
    zorder=3,
)
ax.plot(z_cm, exposure_us, color="0.7", lw=1, zorder=1)

ax.set_yscale("log")
ax.set_xlabel("sensor z-position after axicon3 (cm)")
ax.set_ylabel("calibrated exposure time (µs)")
ax.set_title(
    "Calibrated exposure time vs z-position after axicon #3, "
    "manual XY scans 2026-07-13.\n" + OPTIC_TITLE,
    fontsize=10,
)
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)
fig.colorbar(scatter, ax=ax, label="frames in run (beam larger than sensor → more frames)")

# The exposure calibration targets a fixed peak level, so exposure ∝ 1/peak
# irradiance: the minimum marks where the beam is most concentrated.
i_min = int(np.argmin(exposure_us))
ax.annotate(
    f"min: {exposure_us[i_min]:.0f} µs @ z={z_cm[i_min]:g} cm",
    xy=(z_cm[i_min], exposure_us[i_min]),
    xytext=(z_cm[i_min] - 18, exposure_us[i_min] * 3),
    arrowprops=dict(arrowstyle="->", color="0.3"),
)

plt.tight_layout()
save_fig("calibrated_exposure_time_vs_z")
plt.show()

print(f"exposure range: {exposure_us.min():.0f} – {exposure_us.max():.0f} µs "
      f"(x{exposure_us.max() / exposure_us.min():.0f})")

The calibration holds the *peak* pixel near a fixed target, so exposure time is
essentially a proxy for 1/(peak irradiance): the minimum near z ≈ 150 cm is where
the Bessel core is most intense, and the >200× spread across z is exactly why a
single global exposure could not cover this dataset. Note the repeated z
positions (100 cm and 170 cm were scanned twice, with different exposures) —
they are useful linearity checks below.

In [ ]:
# Per-frame statistics: saturation check + total counts.
# Mono8 saturates at 255; the exposure calibration targeted a peak around
# 0.70 * 255 ≈ 178, so nothing should be near saturation — verify.

SATURATION_LEVEL = 255

records = []
for r in runs:
    for frame_path in r["frame_paths"]:
        frame = np.load(frame_path).astype(np.float64)
        records.append(
            {
                "z_cm": r["z_cm"],
                "exposure_us": r["exposure_us"],
                "max": frame.max(),
                "saturated_px": int(np.sum(frame >= SATURATION_LEVEL)),
                "total_counts": frame.sum(),
                "total_rate": frame.sum() / r["exposure_us"],  # counts / µs
            }
        )

n_saturated_frames = sum(rec["saturated_px"] > 0 for rec in records)
print(f"{len(records)} frames; {n_saturated_frames} contain saturated pixels")
print(f"max pixel value over all frames: {max(rec['max'] for rec in records):.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

rec_z = np.array([rec["z_cm"] for rec in records])
rec_max = np.array([rec["max"] for rec in records])
rec_rate = np.array([rec["total_rate"] for rec in records])

ax1.scatter(rec_z, rec_max, s=25, alpha=0.7)
ax1.axhline(SATURATION_LEVEL, color="r", ls="--", label="saturation (255)")
ax1.axhline(0.70 * 255, color="orange", ls=":", label="calibration target (~178)")
ax1.set_xlabel("z (cm)")
ax1.set_ylabel("max pixel value in frame")
ax1.set_title("Peak level per frame")
ax1.legend()
ax1.minorticks_on()
ax1.grid(True, which="both", alpha=0.3)

ax2.scatter(rec_z, rec_rate, s=25, alpha=0.7)
ax2.set_xlabel("z (cm)")
ax2.set_ylabel("total counts / exposure (counts/µs)")
ax2.set_title("Per-frame total count rate\n(constant iff full beam captured per frame)")
ax2.set_yscale("log")
ax2.minorticks_on()
ax2.grid(True, which="both", alpha=0.3)

fig.suptitle(
    "Per-frame statistics, manual XY scans 2026-07-13.\n" + OPTIC_TITLE,
    fontsize=10,
)

plt.tight_layout()
save_fig("frame_peak_level_and_total_rate_vs_z")
plt.show()

Reading the right-hand plot: if a single frame captured the whole beam and
counts were linear in exposure, `total counts / exposure` would be flat in z
(total power through every z-plane is conserved). Where the beam outgrows the
sensor (small/large z — the multi-frame runs), a *single* frame's total misses
power, so the per-frame rate drops. Any total-power normalization must
therefore operate on the **stitched composite** (with overlap handling), not on
raw frames.

In [ ]:
# Are stitched composites comparable to single frames?
#
# stitcher.stitch_frames feather-AVERAGES overlapping frames
# (composite = sum(frame * feather) / sum(feather)), so composite pixels are
# in the same counts units as raw frames and each scene location is counted
# once. If so, sum(composite) / exposure should land in the same flat band
# as the single-frame runs.

single_rates, comp_rows = [], []
for r in runs:
    T = r["exposure_us"]
    composite_path = r["run_dir"] / "composite.npy"
    if composite_path.exists():
        composite = np.load(composite_path).astype(np.float64)
        comp_rows.append((r["z_cm"], len(r["frame_paths"]), composite.sum() / T))
    elif len(r["frame_paths"]) == 1:
        frame = np.load(r["frame_paths"][0]).astype(np.float64)
        single_rates.append((r["z_cm"], frame.sum() / T))

sz, sr = np.array(single_rates).T
cz, cn, cr = np.array(comp_rows).T

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(sz, sr, xerr=Z_ERR_CM, fmt="o", ms=6, capsize=2, label="single frame")
ax.errorbar(cz, cr, xerr=Z_ERR_CM, fmt="s", ms=7, capsize=2, label="stitched composite")
band = np.median(sr[sz >= 105])
ax.axhline(band, color="0.5", ls="--", label=f"single-frame flat median ({band:.0f})")
for z, n, rate in comp_rows:
    ax.annotate(f"{int(n)}f", (z, rate), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.set_xlabel("z (cm)")
ax.set_ylabel("total counts / exposure (counts/µs)")
ax.set_title(
    "Total count rate: stitched composites vs single-frame runs.\n" + OPTIC_TITLE,
    fontsize=10,
)
ax.legend()
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
save_fig("total_count_rate_vs_z__composite_vs_single_frame")
plt.show()

for z, n, rate in sorted(comp_rows):
    print(f"z={z:6.1f} cm  {int(n):2d} frames  composite rate {rate:7.0f} "
          f"({100 * (rate / band - 1):+.1f}% vs single-frame median)")

Because the stitcher **averages** overlapping frames (rather than summing
them), `composite.npy` is directly comparable to single-frame `.npy` data:
same per-pixel count units, each scene location counted once. The composite
total rates land within a few percent of the single-frame flat band —
conservation of total power holds across the stitched low-z runs too.

Residual caveats: (1) the composite canvas covers more background area, so
its raw total is biased slightly high (~0.2–0.4 counts/px of background ×
a canvas 1.3–1.8× larger); subtract a background level over *covered* pixels
before precision comparisons. (2) Uncovered canvas pixels are exactly 0 —
harmless for sums, but means/backgrounds need a coverage mask (reconstruct
from `composite_offsets.json`; the stitcher does not save its Coverage
array). (3) Any composite that deviates several percent (e.g. the largest
stitch) deserves a registration sanity-check in `composite_offsets.json`
(look for low `OverlapNCC` pairs) before being trusted.

## Normalization: suggested approaches

The Slack summary has the right physical idea — counts should scale linearly
with exposure time — but "scale relative to the longest exposure" is only one
(and not the most robust) way to use it. Suggestions, in recommended order:

### 1. Per-pixel rate normalization (recommended baseline)

Convert every frame to a count *rate* before any comparison:

```
rate(x, y) = (counts(x, y) − dark_offset) / ExposureTime_µs      # counts/µs
```

With Gain = 0 dB and gamma disabled, `rate` is proportional to irradiance at
each pixel, and every z-slice becomes directly comparable with no reference
exposure at all. Scaling "as a fraction of the longest exposure"
(× T/T_max) is this same operation up to one global constant — dividing by T
is simpler and avoids privileging one run. Practical details: cast to float
first, mask any pixel ≥ 255 as invalid (none today, per the check above), and
measure `dark_offset` rather than assuming 0.

### 2. Measure the dark offset & verify linearity (before trusting #1)

The linear model is really `counts = rate × T + offset(T)`. Two cheap checks:

- **Dark frames:** block the beam and capture at several exposures spanning the
  dataset range (0.2 ms → 50 ms). The intercept/level gives `dark_offset`; any
  slope is dark current (should be negligible at these exposures).
- **Repeated-z runs:** z = 100 cm and z = 170 cm were each scanned twice with
  different exposures (e.g. 1635 µs vs 2158 µs at 170 cm). After dividing by
  exposure, their images should agree pixel-for-pixel; disagreement measures
  offset + laser drift. Even better: at 2–3 fixed z, deliberately capture at
  ×0.5, ×1, ×2 of the calibrated exposure and fit counts vs T per pixel.

### 3. Total-power (per-slice) normalization — the Slack/"Shuo" version

Normalize each z-slice by its own integrated signal:
`p(x, y) = rate(x, y) / Σ rate` — each slice becomes a fractional power
distribution, exploiting conservation of total beam power through every
z-plane. This additionally cancels shot-to-shot laser power drift (which #1
does not). Caveats:

- Valid only if the scan captures (essentially) the **whole** beam at that z.
  Today's runs use 1–11 frames per z; totals must come from the stitched
  composite with overlaps counted once, not from summing raw frames.
- At short exposures the faint outer Bessel rings sit near the noise floor and
  bias the total low; subtract a background level estimated from beam-free
  corners before integrating.
- It silently absorbs *real* power loss (clipping on optic apertures), which
  you may actually want to see.

### 4. Use total rate vs z as a data-quality metric

Whichever normalization is primary, plot `Σ counts / T` (per stitched slice)
against z: it should be constant. Deviations flag exactly where an assumption
broke — incomplete spatial coverage, saturation, background, or drift — i.e.
which z positions need a rescan. This turns the normalization assumption into
a measurable check instead of an article of faith.

**Suggested plan:** adopt #1 immediately (it needs nothing new), do the #2
checks on the existing repeated-z runs plus one set of dark frames next lab
session, and layer #3 on top only after the stitched composites pass the #4
flatness check.

## Implementation: per-pixel rate normalization (approach #1)

`rate(x, y) = (counts(x, y) − background(T)) / T` in counts/µs, applied to the
single frame where one frame captured the whole beam, and to `composite.npy`
for the stitched runs (the stitcher feather-averages overlaps, so composite
pixels are in the same count units as raw frames).

In [ ]:
# --- Background model -------------------------------------------------------
# Dark signal model, per pixel (assumed spatially uniform for now):
#
#     counts_dark(T) = offset + dark_rate * T
#
# TODO(leigh, 2026-07-14): record background runs — beam blocked, same room
# lighting / camera temperature — at ~4 exposures spanning the dataset range
# (e.g. 0.2 ms, 2 ms, 20 ms, 51 ms). Save them like normal dataset runs, then
# set BACKGROUND_RUN_GLOB below and re-run this cell to fit the model.
# Until then both coefficients are 0 and normalization is background-free.

BACKGROUND_RUN_GLOB = None  # TODO: e.g. "manual_scan-2026-07-14_*" once recorded

BACKGROUND_OFFSET_COUNTS = 0.0  # counts
BACKGROUND_DARK_RATE = 0.0      # counts/µs

if BACKGROUND_RUN_GLOB is not None:
    bg_exposures, bg_means = [], []
    for run_dir in sorted(DATA_ROOT.glob(BACKGROUND_RUN_GLOB)):
        setup = json.loads((run_dir / "sweep_setup.json").read_text())
        for path in run_dir.glob("*.npy"):
            if path.name.startswith("composite"):
                continue
            bg_exposures.append(setup["CalibratedExposure_us"])
            bg_means.append(float(np.load(path).mean()))

    BACKGROUND_DARK_RATE, BACKGROUND_OFFSET_COUNTS = np.polyfit(
        bg_exposures, bg_means, 1
    )

    plt.figure(figsize=(6, 4))
    plt.scatter(bg_exposures, bg_means, label="dark frame means")
    t = np.linspace(0, max(bg_exposures), 100)
    plt.plot(t, BACKGROUND_OFFSET_COUNTS + BACKGROUND_DARK_RATE * t, "r-",
             label=f"fit: {BACKGROUND_OFFSET_COUNTS:.2f} + {BACKGROUND_DARK_RATE:.2e}·T")
    plt.xlabel("exposure (µs)")
    plt.ylabel("mean dark counts")
    plt.legend()
    plt.minorticks_on()
    plt.grid(True, which="both", alpha=0.3)
    save_fig("dark_counts_vs_exposure_time")
    plt.show()


def background_counts(exposure_us: float) -> float:
    return BACKGROUND_OFFSET_COUNTS + BACKGROUND_DARK_RATE * exposure_us


print(f"background model: {BACKGROUND_OFFSET_COUNTS:.3f} counts "
      f"+ {BACKGROUND_DARK_RATE:.3e} counts/µs · T"
      + ("   (TODO: placeholder — no dark data yet)"
         if BACKGROUND_RUN_GLOB is None else ""))

In [ ]:
# --- Coverage mask + rate normalization --------------------------------------
# The composite canvas is the bounding box of all placed frames, so canvas
# regions never covered by any frame are filled with exactly 0.0 — "no data",
# indistinguishable by value from a real pixel that measured 0 counts. The
# stitcher's per-frame placements are recorded in composite_offsets.json, so
# the mask of true-data pixels can be reconstructed exactly.

def load_coverage_mask(run_dir, composite_shape, frame_shape):
    """Boolean mask of composite pixels actually covered by >=1 frame,
    rebuilt from the placements in composite_offsets.json (mirrors the
    integer rounding + min-shift in stitcher.stitch_frames)."""
    offsets = json.loads((run_dir / "composite_offsets.json").read_text())
    int_offsets = [
        (int(round(o["dy_px"])), int(round(o["dx_px"])))
        for o in offsets["OffsetsPx"]
    ]
    min_dy = min(dy for dy, _ in int_offsets)
    min_dx = min(dx for _, dx in int_offsets)

    mask = np.zeros(composite_shape, dtype=bool)
    h, w = frame_shape
    for dy, dx in int_offsets:
        y, x = dy - min_dy, dx - min_dx
        mask[y : y + h, x : x + w] = True
    return mask


def normalized_rate_image(run):
    """Background-subtracted count-rate image (counts/µs, float64).
    Uncovered composite pixels are NaN so they never enter statistics."""
    T = run["exposure_us"]
    composite_path = run["run_dir"] / "composite.npy"

    if composite_path.exists():
        img = np.load(composite_path).astype(np.float64)
        frame_shape = np.load(run["frame_paths"][0], mmap_mode="r").shape
        mask = load_coverage_mask(run["run_dir"], img.shape, frame_shape)
    else:
        img = np.load(run["frame_paths"][0]).astype(np.float64)
        mask = np.ones(img.shape, dtype=bool)

    rate = (img - background_counts(T)) / T
    rate[~mask] = np.nan
    return rate


# Sanity check on the largest stitch: every out-of-coverage pixel must be
# exactly 0 in the composite (they were never touched by any frame).
check_run = max(runs, key=lambda r: len(r["frame_paths"]))
comp = np.load(check_run["run_dir"] / "composite.npy")
fshape = np.load(check_run["frame_paths"][0], mmap_mode="r").shape
mask = load_coverage_mask(check_run["run_dir"], comp.shape, fshape)
assert np.all(comp[~mask] == 0.0), "coverage mask disagrees with composite"
print(f"z={check_run['z_cm']:g} cm composite: canvas {comp.shape}, "
      f"{mask.mean():.0%} covered, {(~mask).sum()} no-data pixels "
      f"(all exactly 0 in composite — mask verified)")

plt.figure(figsize=(6, 4.5))
plt.imshow(mask, cmap="gray", interpolation="nearest")
plt.title(f"coverage mask, z={check_run['z_cm']:g} cm "
          f"({len(check_run['frame_paths'])} frames)\n"
          "white = real data, black = uncovered canvas (0.0 in composite)")
plt.tight_layout()
save_fig("composite_coverage_mask_xy")
plt.show()

In [ ]:
from matplotlib.colors import LogNorm

# Normalized slices on ONE shared intensity scale — the point of rate
# normalization: slices taken at 226 µs and 51,000 µs become comparable.
demo_zs = [80.0, 130.0, 150.0]
demo_runs = [next(r for r in runs if r["z_cm"] == z) for z in demo_zs]
demo_rates = [normalized_rate_image(r) for r in demo_runs]

vmax = max(np.nanmax(rate) for rate in demo_rates)
norm = LogNorm(vmin=vmax / 3e3, vmax=vmax, clip=True)

fig, axes = plt.subplots(1, len(demo_runs), figsize=(15, 4.6))
for ax, run, rate in zip(axes, demo_runs, demo_rates):
    cmap = plt.get_cmap("inferno").copy()
    cmap.set_bad("0.4")  # NaN = uncovered canvas
    im = ax.imshow(rate, cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_title(f"z={run['z_cm']:g} cm  (T={run['exposure_us']:.0f} µs)\n"
                 f"peak {np.nanmax(rate):.3g} counts/µs")
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="counts/µs (log scale, shared)")
fig.suptitle(
    "Rate-normalized XY slices after axicon #3, shared log scale.\n" + OPTIC_TITLE,
    fontsize=10,
)
save_fig("normalized_xy_slices__shared_log_scale")
plt.show()

# Peak rate vs z: with counts linear in exposure, this is (proportional to)
# the on-axis intensity profile of the Bessel beam.
peak_rates = np.array([np.nanmax(normalized_rate_image(r)) for r in runs])

plt.figure(figsize=(9, 5))
plt.errorbar(z_cm, peak_rates, xerr=Z_ERR_CM, fmt="o-", ms=5, capsize=2)
plt.yscale("log")
plt.xlabel("z (cm)")
plt.ylabel("peak pixel rate (counts/µs)")
plt.title(
    "Measured on-axis intensity (peak count rate) vs z-position "
    "after axicon #3.\n" + OPTIC_TITLE,
    fontsize=10,
)
plt.minorticks_on()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
save_fig("peak_count_rate_vs_z")
plt.show()

Notes on the normalized data:

- The peak-rate curve is the payoff: because the calibration pinned every
  peak near ~178 counts, the raw images carry almost no axial-intensity
  information — it all lives in the exposure times. After normalization the
  ×226 axial intensity variation reappears, now in consistent counts/µs.
- Once tomorrow's dark data is in, re-run from the background cell down:
  everything below picks up the fitted model automatically. If the dark
  frames show spatial structure (vignetting, glow), replace the scalar
  `background_counts(T)` with a per-pixel dark image interpolated in T.
- NaN-aware reducers (`np.nanmax`, `np.nansum`, `np.nanpercentile`) must be
  used on these rate images — plain `np.sum` propagates NaN from the
  uncovered canvas regions.

## Comparison against the Fourier-optics model

The three-axicon QDHT model below is copied verbatim (constants + functions)
from `axicon_bessel_beam_fourier_optics_2026_07_13.ipynb` (§3, §10–11), then
driven with the **as-recorded parameters from `sweep_setup.json`** instead of
that notebook's example values. The measured axial profile (peak count rate
vs z, from the normalization section above) is overlaid on the model's
on-axis intensity, both normalized to their own peaks — this compares the
window's *position and shape*, not absolute intensity.

Two assumptions to confirm at the bench:

- `GaussianBeamWaist_mm = 4.59` is interpreted as the 1/e² **diameter**
  (`WAIST_IS_RADIUS = False` below). Both readings were tried: with the
  *radius* reading (D = 9.18 mm) the model's annulus cannot form cleanly
  within L₁₂ = 190 mm (it needs ≳ 3·zmax ≈ 340 mm) and the predicted window
  lands near 1.0 m — 1.5× off the data; with the *diameter* reading the
  predicted peak lands within 5% of the measurement. **TODO: measure the
  actual 1/e² beam size to settle this** — it also decides whether the real
  annulus at axicon 3 is a clean ring (quick check with a beam card).
- λ = 650 nm and n = 1.4585 are inherited from the model notebook.
  **TODO: confirm the actual laser wavelength.** (The window geometry depends
  on λ only through n(λ), a ~1% effect — not enough to move the window much.)

In [ ]:
# --- Fourier-optics model ----------------------------------------------------
# Copied from axicon_bessel_beam_fourier_optics_2026_07_13.ipynb (§3, §10-11).
# Keep in sync manually; extract to simulator/ if this happens a third time.
from scipy.special import j0 as bessel_j0, j1 as bessel_j1, jn_zeros

LAMBDA = WAVELENGTH_NM * 1e-9   # wavelength [m]  (TODO: confirm laser)
N_AXICON = 1.4585               # fused silica @ ~650 nm
K0 = 2 * np.pi / LAMBDA
mm = 1e-3


class QDHT:
    '''Quasi-discrete Hankel transform of order 0 on r in [0, R].'''
    _cache = {}   # N -> (zeros, jN1, J1abs, T); T is independent of R

    def __init__(self, N, R):
        self.N, self.R = N, R
        if N not in QDHT._cache:
            jz = jn_zeros(0, N + 1)
            jN1 = jz[N]
            jj = jz[:N]
            J1a = np.abs(bessel_j1(jj))
            T = np.empty((N, N))          # built in row blocks to limit peak memory
            step = 1024
            for i0 in range(0, N, step):
                i1 = min(i0 + step, N)
                T[i0:i1] = ((2.0 / jN1) * bessel_j0(np.outer(jj[i0:i1], jj) / jN1)
                            / np.outer(J1a[i0:i1], J1a))
            QDHT._cache[N] = (jj, jN1, J1a, T)
        self.j, self.jN1, self.J1, self.T = QDHT._cache[N]
        self.r = self.j * (R / self.jN1)      # radial grid [m]
        self.kr = self.j / R                  # radial wavevector grid [rad/m]
        self.kr_max = self.jN1 / R

    def forward(self, f):
        return self.T @ (f / self.J1)

    def backward(self, y):
        return (self.T @ y) * self.J1

    def eval_matrix(self, r_eval):
        return (2.0 / self.jN1) * bessel_j0(np.outer(self.kr, r_eval)) / self.J1[:, None]

    def onax(self, Y):
        '''|E(0)|^2 for spectrum rows Y. NB: an extrapolation — r=0 is not a grid point.'''
        return np.abs((2.0 / self.jN1) * (Y @ (1.0 / self.J1))) ** 2

    def power(self, f):
        return (2.0 * self.R**2 / self.jN1**2) * np.sum(np.abs(f) ** 2 / self.J1**2)

    def power_spec(self, y):
        return (2.0 * self.R**2 / self.jN1**2) * np.sum(np.abs(y) ** 2)


def propagate_spectrum(y0, kr, k, z):
    '''Angular-spectrum propagation: multiply by exp(i z sqrt(k^2 - kr^2)).'''
    kz = np.sqrt(np.maximum(k * k - kr * kr, 0.0))
    z = np.atleast_1d(np.asarray(z, dtype=float))
    Y = y0[None, :] * np.exp(1j * kz[None, :] * z[:, None])
    return Y[0] if Y.shape[0] == 1 else Y


def axicon_kr(alpha_deg, n=N_AXICON, k=K0):
    '''Thin-element radial wavevector kr = k (n-1) tan(alpha).'''
    return k * (n - 1) * np.tan(np.radians(alpha_deg))


def simulate_axicon_pair(D, alpha_deg, L_sep, z_after=300 * mm, N=4096,
                         Nz1=140, Nz2=220, pad=None, verbose=True):
    '''Gaussian -> axicon 1 -> L_sep -> axicon 2 -> free space (identical thin axicons).'''
    w0 = D / 2.0
    kr_ax = axicon_kr(alpha_deg)
    theta = kr_ax / K0
    zmax = w0 / theta
    R_ring = theta * L_sep
    R_dom = R_ring + (5 * w0 if pad is None else pad)
    q = QDHT(N, R_dom)
    fringe = 2 * np.pi / kr_ax
    dr = q.r[1] - q.r[0]
    if verbose:
        print(f"alpha={alpha_deg} x2  D={D/mm:g} mm  L={L_sep/mm:g} mm | "
              f"zmax={zmax/mm:6.1f} mm  ring={R_ring/mm:5.2f} mm | "
              f"{fringe/dr:4.1f} pts/fringe, kr_max/kr={q.kr_max/kr_ax:4.1f}, "
              f"wall margin={(R_dom-R_ring)/w0:4.1f} w0")
    assert L_sep > 1.2 * zmax, "separation should exceed zmax for a clean annulus"
    assert q.kr_max > 1.5 * kr_ax and fringe / dr > 2.5

    t_ax = np.exp(-1j * kr_ax * q.r)
    E0 = np.exp(-(q.r / w0) ** 2)
    y1 = q.forward(E0 * t_ax)
    z1 = np.linspace(0.0, L_sep, Nz1)
    Y1 = propagate_spectrum(y1, q.kr, K0, z1)
    E_at2 = q.backward(Y1[-1])
    y2 = q.forward(E_at2 * t_ax)
    z2 = np.linspace(0.0, z_after, Nz2)
    Y2 = propagate_spectrum(y2, q.kr, K0, z2)
    return dict(D=D, w0=w0, alpha_deg=alpha_deg, kr=kr_ax, theta=theta, zmax=zmax,
                L_sep=L_sep, R_ring=R_ring, q=q, z1=z1, z2=z2, Y1=Y1, Y2=Y2,
                onax2=q.onax(Y2), P_in=q.power(E0), P_out=q.power_spec(y2))


def ring_at(pair_sim, L23):
    '''Field, ring peak radius, and ring FWHM at plane L23 after axicon 2.'''
    q = pair_sim["q"]
    iz = np.argmin(np.abs(pair_sim["z2"] - L23))
    E_in = q.backward(pair_sim["Y2"][iz])
    I = np.abs(E_in) ** 2
    i0 = np.searchsorted(q.r, 0.4 * pair_sim["R_ring"])
    j = i0 + np.argmax(I[i0:])
    half = 0.5 * I[j]
    lo = np.where(I[i0:j] < half)[0]
    hi = np.where(I[j:] < half)[0]
    wid = q.r[j + hi[0]] - q.r[i0 + lo[-1]] if len(lo) and len(hi) else np.nan
    return E_in, q.r[j], wid


def simulate_third_axicon(pair_sim, alpha3_deg, L23=100 * mm, Nz3=300, verbose=True):
    '''Apply axicon 3 in the annulus (L23 after axicon 2) and propagate.'''
    q = pair_sim["q"]
    kr3 = axicon_kr(alpha3_deg)
    th3 = kr3 / K0
    E_in, R_a, Delta = ring_at(pair_sim, L23)
    I_ring = np.abs(E_in) ** 2
    y3 = q.forward(E_in * np.exp(-1j * kr3 * q.r))
    z3 = np.linspace(1e-3, 1.3 * (R_a + Delta) / th3, Nz3)
    Y3 = propagate_spectrum(y3, q.kr, K0, z3)
    onax = q.onax(Y3)
    onax_th = 2 * np.pi * kr3**2 / K0 * z3 * np.interp(th3 * z3, q.r, I_ring)
    z_wall = (q.R + R_a) / th3
    if verbose:
        print(f"alpha3={alpha3_deg}: ring R_a={R_a/mm:.2f} mm, "
              f"Delta={Delta/mm:.2f} mm -> zone "
              f"{(R_a-Delta/2)/th3:.2f}-{(R_a+Delta/2)/th3:.2f} m | "
              f"wall headroom={z_wall/z3[-1]:.1f}x")
    return dict(alpha3=alpha3_deg, kr3=kr3, th3=th3, R_a=R_a, Delta=Delta,
                z3=z3, y3=y3, Y3=Y3, onax=onax, onax_th=onax_th, q=q,
                P3=q.power_spec(y3))

In [ ]:
# --- Run the model with the as-recorded parameters and overlay the data ------
optic = json.loads((runs[0]["run_dir"] / "sweep_setup.json").read_text())[
    "OpticConfiguration"
]
print("OpticConfiguration:", optic)

WAIST_IS_RADIUS = False  # TODO: confirm — diameter fits the data; see header
D_in = (2 if WAIST_IS_RADIUS else 1) * optic["GaussianBeamWaist_mm"] * mm

assert optic["Axicon1_deg"] == optic["Axicon2_deg"], (
    "the pair model assumes identical axicons 1 and 2"
)

pair = simulate_axicon_pair(
    D=D_in,
    alpha_deg=optic["Axicon1_deg"],
    L_sep=optic["L12_mm"] * mm,
    z_after=2 * optic["L23_mm"] * mm,
    N=4096,
    Nz1=8,          # only the final plane of leg 1 is used
    Nz2=40,
    pad=2.5 * D_in / 2,   # tighter domain than the default so the radial
                          # grid still resolves the axicon fringes at this D
)
model = simulate_third_axicon(pair, optic["Axicon3_deg"], L23=optic["L23_mm"] * mm)


def peak_and_fwhm(z, y):
    i = int(np.argmax(y))
    half = 0.5 * y[i]
    lo = np.where(y[:i] < half)[0]
    hi = np.where(y[i:] < half)[0]
    z_lo = z[lo[-1]] if len(lo) else z[0]
    z_hi = z[i + hi[0]] if len(hi) else z[-1]
    return z[i], z_hi - z_lo


z_model_cm = model["z3"] * 100.0
model_norm = model["onax"] / model["onax"].max()
meas_norm = peak_rates / peak_rates.max()

zpk_model, fwhm_model = peak_and_fwhm(z_model_cm, model["onax"])
zpk_meas, fwhm_meas = peak_and_fwhm(z_cm, peak_rates)

fig, ax = plt.subplots(figsize=(9.5, 5))
ax.plot(z_model_cm, model_norm, "-", color="tab:orange", lw=1.6,
        label="QDHT model")
ax.errorbar(z_cm, meas_norm, xerr=Z_ERR_CM, fmt="o", ms=5, capsize=2,
            color="tab:blue", label="measured peak rate (±5 mm in z)")
ax.set_xlabel("z after axicon3 (cm)")
ax.set_ylabel("on-axis intensity, normalized to peak")
ax.set_xlim(min(z_cm.min(), 50), max(z_cm.max(), z_model_cm.max()) + 10)
ax.set_title(
    "Measured on-axis intensity vs Z-position after axicon #3.\n"
    f"Comparison with {QDHT_ATTRIBUTION}\n" + OPTIC_TITLE,
    fontsize=10,
)
ax.legend()
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
save_fig("on_axis_intensity_vs_z__QDHT_comparison")
plt.show()

print(f"measured : peak z = {zpk_meas:6.1f} cm   FWHM = {fwhm_meas:5.1f} cm")
print(f"model    : peak z = {zpk_model:6.1f} cm   FWHM = {fwhm_model:5.1f} cm")
print(f"peak-position ratio measured/model = {zpk_meas / zpk_model:.3f}")
print(f"-> effective alpha3 if geometry explains the shift alone: "
      f"{optic['Axicon3_deg'] * zpk_model / zpk_meas:.3f} deg "
      f"(nominal {optic['Axicon3_deg']:g} deg)")

Interpreting the comparison:

- **Window center** in the model is set almost purely by geometry:
  z ≈ R_a/tan θ₃ with R_a ≈ (n−1)tan(α₁₂)·L₁₂, i.e. roughly
  tan(α₁₂)/tan(α₃) × L₁₂ ≈ 10 × 19 cm ≈ 1.9 m for the nominal parameters —
  independent of the beam waist and (to first order) of λ. If the measured
  peak sits at noticeably different z, the prime suspects are, in order:
  the **α₃ apex-angle tolerance** (the window position scales as 1/tan α₃, so
  a 0.5° axicon at the loose end of a ±0.1° tolerance moves the window by
  ±20%), the effective **L₁₂** (measured face-to-face vs principal planes),
  and apex rounding on axicon 3. The printed "effective α₃" converts the
  peak-position ratio into the apex angle that would explain it.
- **Window length** (FWHM) scales with the annulus width ≈ w₀. With the
  diameter reading the model FWHM comes out somewhat wider than measured —
  consistent with the beam being slightly smaller than 4.59 mm, or with the
  measured profile under-resolving the window tails at 5 cm sampling.
- The measured curve is sampled every ~5 cm with ±0.5 cm z uncertainty, so
  fine ripple structure in the model cannot be resolved — compare envelope,
  center, and width only.
- The z=140 cm dip in the measured profile mirrors its anomalous exposure
  calibration; recheck that point before reading physics into it.

## Central-lobe width vs z

Width of the on-axis Bessel core: at every z, take the (normalized) image,
find the brightest pixel, azimuthally average the intensity about it, and
measure the FWHM of that radial profile. Inside the Bessel window the model
says the core FWHM is set by α₃ alone (ideal J₀ core: first zero at
2.4048/k_r3, FWHM ≈ 2.25/k_r3) and is nearly independent of z — so this is an
independent check on the axicon-3 angle, complementary to the window
position above.

Caveat: **outside** the window there is no on-axis core, so the
"brightest lobe" measures whatever is brightest there instead. In this
dataset: below ~110 cm it locks onto a sharp fringe of the *converging
annulus* (25–38 µm — even narrower than the core, and growing smoothly as
the cone converges), and past ~180 cm onto the diverging ring system
(mm scale). Only the points between the two step changes (~115–180 cm)
measure the Bessel core — the steps themselves mark the window edges.

In [ ]:
# --- Central-lobe FWHM, measured ---------------------------------------------
PIXEL_UM = 3.45  # BFS-PGE-31S4M pixel pitch (Sony IMX265, 2048 x 1536)


def radial_profile_about_peak(img, r_max_px):
    '''Azimuthally averaged profile about the centroid-refined brightest
    pixel. NaN-aware (stitched composites). Returns (r_px_centers, profile).'''
    iy, ix = np.unravel_index(np.nanargmax(img), img.shape)

    # Centroid refinement over a small window around the peak pixel.
    y0, y1 = max(iy - 3, 0), min(iy + 4, img.shape[0])
    x0, x1 = max(ix - 3, 0), min(ix + 4, img.shape[1])
    win = np.nan_to_num(img[y0:y1, x0:x1], nan=0.0)
    yy, xx = np.mgrid[y0:y1, x0:x1]
    cy = float((win * yy).sum() / win.sum())
    cx = float((win * xx).sum() / win.sum())

    # Crop to the region of interest and bin by integer radius (1 px bins).
    y0, y1 = int(max(cy - r_max_px, 0)), int(min(cy + r_max_px + 1, img.shape[0]))
    x0, x1 = int(max(cx - r_max_px, 0)), int(min(cx + r_max_px + 1, img.shape[1]))
    crop = img[y0:y1, x0:x1]
    yy, xx = np.mgrid[y0:y1, x0:x1]
    r_px = np.hypot(yy - cy, xx - cx)

    valid = ~np.isnan(crop) & (r_px < r_max_px)
    rbin = r_px[valid].astype(int)
    sums = np.bincount(rbin, weights=crop[valid], minlength=r_max_px)
    counts = np.bincount(rbin, minlength=r_max_px)
    profile = np.divide(sums, counts, out=np.full(len(sums), np.nan),
                        where=counts > 0)[:r_max_px]
    return np.arange(r_max_px) + 0.5, profile


def fwhm_of_profile(r_px, profile):
    '''Full width (px) at half the central value: first half-max crossing,
    linearly interpolated. NaN if the profile never falls below half.'''
    peak = np.nanmax(profile[:3])   # central value (bin 0 alone can be noisy)
    half = 0.5 * peak
    below = np.where(profile < half)[0]
    if len(below) == 0 or below[0] == 0:
        return np.nan
    i = below[0]
    frac = (profile[i - 1] - half) / (profile[i - 1] - profile[i])
    return 2.0 * (r_px[i - 1] + frac * (r_px[i] - r_px[i - 1]))


lobe_fwhm_um = []
for r in runs:
    rate = normalized_rate_image(r)
    r_px, profile = radial_profile_about_peak(rate, r_max_px=700)
    lobe_fwhm_um.append(fwhm_of_profile(r_px, profile) * PIXEL_UM)
lobe_fwhm_um = np.array(lobe_fwhm_um)

for z, w in zip(z_cm, lobe_fwhm_um):
    print(f"z={z:6.1f} cm   brightest-lobe FWHM = {w:8.1f} um")

In [ ]:
# --- Model core width + comparison plot ---------------------------------------
from scipy.optimize import brentq

# Ideal J0 Bessel core for the nominal alpha3.
x_half = brentq(lambda x: bessel_j0(x) ** 2 - 0.5, 0.5, 2.0)
fwhm_j0_um = 2.0 * x_half / model["kr3"] * 1e6
r_zero_um = 2.4048 / model["kr3"] * 1e6
print(f"ideal J0 core (alpha3={model['alpha3']:g} deg): "
      f"FWHM = {fwhm_j0_um:.1f} um, first zero at r = {r_zero_um:.1f} um")

# Core FWHM along the window, measured from the QDHT simulation the same way
# (radial profile from r=0; inside the window the on-axis point is the peak).
r_eval = np.linspace(0.0, 300e-6, 400)
I_rz = np.abs(model["Y3"] @ model["q"].eval_matrix(r_eval)) ** 2
in_window = model["onax"] > 0.05 * model["onax"].max()

model_fwhm_um = np.full(len(model["z3"]), np.nan)
for i in np.where(in_window)[0]:
    row = I_rz[i]
    below = np.where(row < 0.5 * row[0])[0]
    if len(below) and below[0] > 0:
        j = below[0]
        frac = (row[j - 1] - 0.5 * row[0]) / (row[j - 1] - row[j])
        model_fwhm_um[i] = 2.0 * (r_eval[j - 1]
                                  + frac * (r_eval[j] - r_eval[j - 1])) * 1e6

fig, ax = plt.subplots(figsize=(9.5, 5))
ax.errorbar(z_cm, lobe_fwhm_um, xerr=Z_ERR_CM, fmt="o", ms=5, capsize=2,
            label="measured: brightest-lobe FWHM")
ax.plot(z_model_cm, model_fwhm_um, "-", color="tab:orange", lw=1.6,
        label="QDHT model: core FWHM (inside window)")
ax.axhline(fwhm_j0_um, color="0.4", ls=":",
           label=f"ideal J0 core FWHM ({fwhm_j0_um:.0f} µm)")
ax.set_yscale("log")
ax.set_xlabel("z after axicon3 (cm)")
ax.set_ylabel("FWHM (µm)")
ax.set_title(
    "Brightest-lobe FWHM vs z-position after axicon #3 "
    "(out-of-window points = ring/fringe, not core).\n"
    f"Comparison with {QDHT_ATTRIBUTION}\n" + OPTIC_TITLE,
    fontsize=10,
)
ax.legend()
ax.minorticks_on()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
save_fig("brightest_lobe_fwhm_vs_z__QDHT_comparison")
plt.show()

in_zone = (z_cm > 115) & (z_cm < 180) & np.isfinite(lobe_fwhm_um)
print(f"measured core FWHM inside the window (115-180 cm): "
      f"median {np.median(lobe_fwhm_um[in_zone]):.1f} um "
      f"(n={in_zone.sum()}), ideal J0: {fwhm_j0_um:.1f} um")

## XY slice galleries: measured vs model

Two grids over the same z positions, both per-slice normalized (each panel
scaled to its own peak, like the reference Bessel–Gauss plot) and cropped to
the same ±4 mm field of view so they can be compared panel-by-panel:

1. **Measured** — normalized rate image at each z, centered on its intensity
   centroid (the beam is not centered on the sensor), NaN-padded where the
   crop extends beyond the frame / stitched coverage. Titles carry the z
   position and acquisition timestamp.
2. **Model** — the QDHT radial profile I(r, z) at the nearest simulated z,
   spun into a 2D XY image.

Per-slice normalization deliberately hides the ×226 axial intensity
variation (that story is the axial-profile plot above); these grids compare
*structure* — ring radius, ring count, core size — not brightness.

In [ ]:
# --- Measured XY slice grid ----------------------------------------------------
HALF_MM = 4.0          # half-width of every panel's field of view
PANEL_PX = 400         # approximate downsampled panel size


def run_timestamp(run):
    '''"2026-07-13 18:57" from a run directory named manual_scan-<date>_<time>.'''
    stamp = run["run_dir"].name.split("manual_scan-")[1]
    date, time = stamp.split("_")
    return f"{date} {time.replace('-', ':')[:5]}"


def centered_slice(rate, half_mm=HALF_MM):
    '''Crop the rate image to +/-half_mm about its intensity centroid,
    NaN-padding beyond the data, downsampled for display.'''
    px_mm = PIXEL_UM / 1000.0
    w = np.nan_to_num(rate, nan=0.0)
    total = w.sum()
    cy = float((w.sum(axis=1) * np.arange(rate.shape[0])).sum() / total)
    cx = float((w.sum(axis=0) * np.arange(rate.shape[1])).sum() / total)

    half_px = int(round(half_mm / px_mm))
    out = np.full((2 * half_px, 2 * half_px), np.nan)

    y0, x0 = int(round(cy)) - half_px, int(round(cx)) - half_px
    ys0, xs0 = max(y0, 0), max(x0, 0)
    ys1 = min(y0 + 2 * half_px, rate.shape[0])
    xs1 = min(x0 + 2 * half_px, rate.shape[1])
    out[ys0 - y0 : ys1 - y0, xs0 - x0 : xs1 - x0] = rate[ys0:ys1, xs0:xs1]

    step = max(1, (2 * half_px) // PANEL_PX)
    return out[::step, ::step]


n_cols = 6
n_rows = int(np.ceil(len(runs) / n_cols))
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("0.15")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.55 * n_cols, 2.75 * n_rows))
for ax in axes.ravel():
    ax.set_axis_off()

for ax, r in zip(axes.ravel(), runs):
    panel = centered_slice(normalized_rate_image(r))
    ax.set_axis_on()
    im = ax.imshow(
        np.sqrt(panel / np.nanmax(panel)),
        cmap=cmap, vmin=0, vmax=1,
        extent=[-HALF_MM, HALF_MM, -HALF_MM, HALF_MM],
        interpolation="nearest",
    )
    ax.set_title(f"z = {r['z_cm']:g} cm\n{run_timestamp(r)}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(
    "Measured XY slices at various z-positions after axicon #3, "
    f"normalized per-panel, sqrt scale, ±{HALF_MM:g} mm about centroid.\n"
    + OPTIC_TITLE,
    fontsize=12, y=0.995,
)
fig.colorbar(im, ax=axes, label="sqrt(normalized intensity)", fraction=0.02, pad=0.01)
save_fig("measured_xy_slices_grid")
plt.show()

In [ ]:
# --- Model XY slice grid --------------------------------------------------------
# Radial profile I(r, z) from the QDHT simulation at the z nearest each
# measurement, spun into a 2D image over the same +/-4 mm field.
r_eval = np.linspace(0.0, 1.6 * HALF_MM * 1e-3, 1500)   # resolves the 58 µm core
M_eval = model["q"].eval_matrix(r_eval)

axis_mm = np.linspace(-HALF_MM, HALF_MM, 401)
xx_mm, yy_mm = np.meshgrid(axis_mm, axis_mm)
rr_m = np.hypot(xx_mm, yy_mm) * 1e-3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.55 * n_cols, 2.75 * n_rows))
for ax in axes.ravel():
    ax.set_axis_off()

for ax, r in zip(axes.ravel(), runs):
    iz = int(np.argmin(np.abs(model["z3"] - r["z_cm"] / 100.0)))
    profile = np.abs(model["Y3"][iz] @ M_eval) ** 2
    panel = np.interp(rr_m, r_eval, profile)
    ax.set_axis_on()
    im = ax.imshow(
        np.sqrt(panel / panel.max()),
        cmap="viridis", vmin=0, vmax=1,
        extent=[-HALF_MM, HALF_MM, -HALF_MM, HALF_MM],
        interpolation="nearest",
    )
    ax.set_title(f"z = {r['z_cm']:g} cm\nmodel (z3 = {model['z3'][iz]*100:.0f} cm)",
                 fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(
    f"{QDHT_ATTRIBUTION} XY slices after axicon #3, "
    f"normalized per-panel, sqrt scale, ±{HALF_MM:g} mm.\n" + OPTIC_TITLE,
    fontsize=12, y=0.995,
)
fig.colorbar(im, ax=axes, label="sqrt(normalized intensity)", fraction=0.02, pad=0.01)
save_fig("qdht_model_xy_slices_grid")
plt.show()